# Fig. 1i — Combined Imputed Marker-Gene Heatmap (All Conditions, Downsampled)

**Purpose:** Build one combined imputed-expression heatmap (marker genes x
predicted cell type) across four conditions/samples:

| library_id            | donor/sample ID                | condition        |
|------------------------|----------------------------------|-------------------|
| QY_2673_1_2            | HL230324_filt                    | MASL              |
| QY_2665_1_2             | HL160029_filt                    | Normal            |
| QY_2480_Liver_MASH      | HL20221019_broad_no_mast_filt    | MASH              |
| QY_2666                | HL170058_filt                    | MASH, fibrosis stage 2 (used in place of MetALD; see note below) |

**Note:** an early attempt merged all four samples' full imputed objects
directly and ran out of memory (imputed `.h5ad` files here are tens to
~90 GB each, and the combined object needs ~700 GB of memory to build). The
notebook instead downsamples each sample's imputed object to 100,000 cells
*before* concatenating.

**Note:** a MetALD sample was originally intended for the 4th condition, but
a NASH/MASH fibrosis-stage-2 sample (`HL170058_filt`) was substituted here.

**Pipeline:** for each sample: load the imputed (`adge_merge.h5ad`) and
Tangram-predicted (`adata_0.5_tangram_filt.h5ad`) objects, subset the
imputed object to the predicted cells and copy over `pred_cell_types` +
spatial coordinates, downsample to 100k cells. Concatenate all four
downsampled objects, then plot a marker-gene x cell-type dot/heatmap of
average imputed expression (marker list here has `thy1` and `aspn` removed
compared to earlier versions).

**Cleanup notes (this pass):** removed a `process_sample` helper function
that was defined but never actually called (each sample below was loaded
with its own near-identical inline block instead, and the helper also called
plotting functions that aren't defined until later in the notebook, so it
couldn't have run as written); removed a color-palette dict that ended up
unused (the final heatmap function takes no `color_map` argument); removed
duplicate/dead cells (bare variable-display debugging cells, an abandoned
duplicate `ad.concat` attempt for only 3 samples, scattered duplicate
imports). No data-processing logic that feeds the final heatmap was
changed.

In [3]:
import os
import datetime

import numpy as np
import scanpy as sc
import anndata as ad
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt


In [4]:
col_map_red = mpl.colormaps['Reds'].resampled(256)


## Configuration

In [9]:
# Base directory for imputed (Tangram) outputs; overridden per-sample below where needed
idir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/deep_seq/'


In [10]:
filt_h5ad_dir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/misc/tangram_scores_dist2/'


In [13]:
odir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/misc/qc_plots_comb/'


## Load, subset, and downsample each sample

Each block below: loads the imputed + Tangram-predicted objects for one sample, subsets the imputed object to the predicted cells, copies over `pred_cell_types` and spatial coordinates, then downsamples to 100k cells.

### MASL — HL230324_filt

In [11]:
sample_nm = 'HL230324_filt'


In [14]:
sample_dir = os.path.join(idir, sample_nm)
adge_file = os.path.join(sample_dir, "merge", "adge_merge.h5ad")

filt_h5ad_sample_dir = os.path.join(filt_h5ad_dir, sample_nm)
ad_sp_file = os.path.join(filt_h5ad_sample_dir, "adata_0.5_tangram_filt.h5ad")
patdir = os.path.join(odir, sample_nm)
os.makedirs(patdir, exist_ok=True)

adge_masl = sc.read_h5ad(adge_file)
adata_pred_filt_masl = sc.read_h5ad(ad_sp_file)

subset_cells_masl = adata_pred_filt_masl.obs_names
adge_filt_masl = adge_masl[adge_masl.obs_names.isin(subset_cells_masl)].copy()

adge_filt_masl.obs['pred_cell_types'] = adata_pred_filt_masl.obs['pred_cell_types']
adge_filt_masl.obsm['spatial'] = adata_pred_filt_masl.obsm['spatial']
adge_filt_masl.uns['spatial'] = adata_pred_filt_masl.uns['spatial']


In [17]:
sc.pp.subsample(adge_filt_masl, n_obs=100000, random_state=42)


### Normal — HL160029_filt

In [19]:
sample_nm = 'HL160029_filt'


In [20]:
sample_dir = os.path.join(idir, sample_nm)
adge_file = os.path.join(sample_dir, "merge", "adge_merge.h5ad")

filt_h5ad_sample_dir = os.path.join(filt_h5ad_dir, sample_nm)
ad_sp_file = os.path.join(filt_h5ad_sample_dir, "adata_0.5_tangram_filt.h5ad")
patdir = os.path.join(odir, sample_nm)
os.makedirs(patdir, exist_ok=True)

adge_normal = sc.read_h5ad(adge_file)
adata_pred_filt_normal = sc.read_h5ad(ad_sp_file)

subset_cells_normal = adata_pred_filt_normal.obs_names
adge_filt_normal = adge_normal[adge_normal.obs_names.isin(subset_cells_normal)].copy()

adge_filt_normal.obs['pred_cell_types'] = adata_pred_filt_normal.obs['pred_cell_types']
adge_filt_normal.obsm['spatial'] = adata_pred_filt_normal.obsm['spatial']
adge_filt_normal.uns['spatial'] = adata_pred_filt_normal.uns['spatial']


In [23]:
sc.pp.subsample(adge_filt_normal, n_obs=100000, random_state=42)


### MASH — HL20221019_broad_no_mast_filt

Note: this sample's imputed output lives under a different base directory (`.../tangram/` rather than `.../tangram/deep_seq/`), so `idir` is reassigned here.

In [26]:
sample_nm = 'HL20221019_broad_no_mast_filt'


In [27]:
idir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/'


In [28]:
sample_dir = os.path.join(idir, sample_nm)
adge_file = os.path.join(sample_dir, "merge", "adge_merge.h5ad")

filt_h5ad_sample_dir = os.path.join(filt_h5ad_dir, sample_nm)
ad_sp_file = os.path.join(filt_h5ad_sample_dir, "adata_0.5_tangram_filt.h5ad")
patdir = os.path.join(odir, sample_nm)
os.makedirs(patdir, exist_ok=True)

adge_mash = sc.read_h5ad(adge_file)
adata_pred_filt_mash = sc.read_h5ad(ad_sp_file)

subset_cells_mash = adata_pred_filt_mash.obs_names
adge_filt_mash = adge_mash[adge_mash.obs_names.isin(subset_cells_mash)].copy()

adge_filt_mash.obs['pred_cell_types'] = adata_pred_filt_mash.obs['pred_cell_types']
adge_filt_mash.obsm['spatial'] = adata_pred_filt_mash.obsm['spatial']
adge_filt_mash.uns['spatial'] = adata_pred_filt_mash.uns['spatial']


In [31]:
sc.pp.subsample(adge_filt_mash, n_obs=100000, random_state=42)


### MASH, fibrosis stage 2 — HL170058_filt (used in place of MetALD)

In [35]:
idir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/tangram/deep_seq/'


In [36]:
sample_nm = 'HL170058_filt'


In [37]:
sample_dir = os.path.join(idir, sample_nm)
adge_file = os.path.join(sample_dir, "merge", "adge_merge.h5ad")

filt_h5ad_sample_dir = os.path.join(filt_h5ad_dir, sample_nm)
ad_sp_file = os.path.join(filt_h5ad_sample_dir, "adata_0.5_tangram_filt.h5ad")
patdir = os.path.join(odir, sample_nm)
os.makedirs(patdir, exist_ok=True)

adge_mash_fibstage2 = sc.read_h5ad(adge_file)
adata_pred_filt_mash_fibstage2 = sc.read_h5ad(ad_sp_file)

subset_cells_mash_fibstage2 = adata_pred_filt_mash_fibstage2.obs_names
adge_filt_mash_fibstage2 = adge_mash_fibstage2[adge_mash_fibstage2.obs_names.isin(subset_cells_mash_fibstage2)].copy()

adge_filt_mash_fibstage2.obs['pred_cell_types'] = adata_pred_filt_mash_fibstage2.obs['pred_cell_types']
adge_filt_mash_fibstage2.obsm['spatial'] = adata_pred_filt_mash_fibstage2.obsm['spatial']
adge_filt_mash_fibstage2.uns['spatial'] = adata_pred_filt_mash_fibstage2.uns['spatial']


In [40]:
sc.pp.subsample(adge_filt_mash_fibstage2, n_obs=100000, random_state=42)


## Concatenate all four downsampled samples

In [45]:
print(datetime.datetime.now())

adge_combined = ad.concat(
    [adge_filt_masl, adge_filt_normal, adge_filt_mash, adge_filt_mash_fibstage2],
    axis=0,
    join='outer',
    uns_merge='unique',
    label='batch',
    keys=['HL230324_filt', 'HL160029_filt', 'HL20221019_broad_no_mast_filt', 'HL170058_filt'],
    index_unique='-',
)

print(datetime.datetime.now())


2025-03-21 12:37:48.457077
2025-03-21 12:40:24.989982


In [49]:
adge_combined.obs['batch'].value_counts()

batch
HL230324_filt                    100000
HL160029_filt                    100000
HL20221019_broad_no_mast_filt    100000
HL170058_filt                    100000
Name: count, dtype: int64

## Marker gene list

Genes commented out below were tried and excluded (not found in the imputed object, or superseded by a related marker), kept here as a record of that process. `thy1` and `aspn` were removed for this version of the heatmap (see notebook title).

In [52]:
brin_markers = [
    "cd19", "ms4a1",
    "krt19", "fxyd2", "spp1",
    "epcam", "sox9", "anxa4",
    # "sry",
    "krt1",
    # "pecam1",  # can't find
    "cd146",
    "mcam",
    # "sele",
    "flt4",
    # "lyve1",
    "cd34",
    # "ptprc",
    "stab2", "ptprb",  # endothelial
    "col1a1",
    # "fap",
    "adamts13",
    "ngfr", "cygb", "hgf", "rbp1",  # stellate cells
    # "msln",
    # "grem1",
    # "calca",
    "eln",  # Gremlin1, Asporin, calcitonin a, Elastin (fibroblasts)
    "cyp2e1", "hnf4a", "crp", "alb",
    "serpina1", "ttr",  # hepatocytes
    "c1qa", "cd163", "timd4",
    "cd68", "ms4a7",  # myeloid
    "cd69", "trbc2", "cd3d",
]


## Output directory for the combined heatmap

In [53]:
patdir = '/tscc/projects/ps-epigen/users/cmiciano/Liver/RNA/spatial/outputs/sandbox/misc/adge_combined4_ds_nash_fibstage2_rm_thy1_aspn/'

## Heatmap function

In [54]:
def create_expression_heatmaps(raw_adata, imputed_adata, markers, groupby, outdir, sample_name):
    """
    Generate a heatmap of average (imputed) gene expression for a set of
    marker genes, grouped by predicted cell type.

    Parameters
    ----------
    raw_adata : AnnData
        Included for interface compatibility (a raw-counts version of this
        heatmap can be produced the same way); not used by this call.
    imputed_adata : AnnData
        Imputed-expression AnnData to plot.
    markers : list[str]
        Marker genes to include (only those present in `imputed_adata.var_names`
        are kept).
    groupby : str
        Column in `.obs` to group cells by (e.g. predicted cell type).
    outdir : str
        Directory to save the output plot into.
    sample_name : str
        Used to label the output filename/plot title.
    """
    os.makedirs(outdir, exist_ok=True)

    def plot_heatmap(adata, dataset_type, markers):
        markers = [gene for gene in markers if gene in adata.var_names]

        # Use scanpy's dotplot machinery to compute per-group average
        # expression (standard-scaled per gene), then re-plot as a heatmap.
        dotplot = sc.pl.dotplot(
            adata,
            var_names=markers,
            groupby=groupby,
            color_map="viridis",
            dot_max=1,
            dot_min=0,
            vmin=0.2,
            expression_cutoff=0.1,
            show=False,
            standard_scale="var",
            return_fig=True,
        )
        avg_expression = dotplot.dot_color_df

        plt.figure(figsize=(14, 20))
        col_map_red = mpl.colormaps['Reds'].resampled(256)
        sns.heatmap(
            avg_expression.T,
            cmap=col_map_red,
            fmt=".2f",
            linewidths=0.5,
            cbar_kws={"label": "Average Expression"},
        )
        plt.xlabel("Genes")
        plt.ylabel("Cell Groups")
        plt.title(f"Heatmap of Average Gene Expression ({dataset_type} Counts) - {sample_name}")
        plt.tight_layout()

        heatmap_filename = os.path.join(outdir, f"{sample_name}_{dataset_type}_heatmap.pdf")
        plt.savefig(heatmap_filename)
        plt.close()

    # Only the imputed-counts heatmap is generated here (a raw-counts version
    # can be produced the same way via plot_heatmap(raw_adata, "raw", markers)).
    plot_heatmap(imputed_adata, "imputed", markers)


## Generate the combined heatmap

In [56]:
sample_nm = 'combined_normal_masl_mash_fibstage2_rm_thy1_aspn'


In [58]:
create_expression_heatmaps(
    raw_adata=adata_pred_filt_masl,  # unused by this call; kept for the function's interface
    imputed_adata=adge_combined,
    markers=brin_markers,
    groupby="pred_cell_types",
    outdir=patdir,
    sample_name=sample_nm,
)
